# Whirlpool Experiment — Master Runner

Executes the full analysis pipeline without visualizations:

1. **Load run data** — attribute tables + wide-format simulation output
2. **Post-processing** — intertemporal decomposition / rescaling
3. **Cost-benefits** — system + technical costs → `cost_benefits_data_whirlpool.csv`
4. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_whirlpool.csv`
5. **Tableau export** — whirlpool plot data → `Tableau/data/tableau_whirlpool.csv`
6. **MAC comparison** — tornado vs whirlpool → `Tableau/data/mac_tornado_to_whirlpool.csv`

All configuration lives in `scripts/config.py`.


## 0 · Environment setup

In [1]:
import os
import sys
import pathlib
import logging
import warnings

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("whirlpool", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir : {cfg.PROJECT_DIR}")
logger.info(f"Run output  : {cfg.RUN_ID_OUTPUT_DIR}")

2026-04-10 15:20:16,487 - INFO - Environment ready.
2026-04-10 15:20:16,488 - INFO - Project dir : /Users/fabianfuentes/git/ssp_libya
2026-04-10 15:20:16,488 - INFO - Run output  : /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336


## 1 · Load run data

In [2]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

2026-04-10 15:20:16,497 - INFO - att_primary  : (70, 4)
2026-04-10 15:20:16,498 - INFO - att_strategy : (145, 10)
2026-04-10 15:20:17,158 - INFO - df_export    : (2520, 4107)


## 2 · Post-processing — intertemporal decomposition

In [3]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_biomass (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_coal (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_coal_ccs (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_gas_ccs (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_geothermal (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_hydropower (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_nuclear (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_ocean (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_solar (time_period == 8)
Changed 70 zero(s) in: emission_co2e_ch4_entc_generation_pp_wind (time_period == 8)
Changed 70 zero(s) in: emission_co2e_co2_entc_generation_pp_coal (time_period == 8)
Changed 70 zero(s) in: emission_co2e_co2_entc_gen

2026-04-10 15:20:29,964 - INFO - df_decomposed : (1960, 4107)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/decomposed_ssp_output_whirlpool.csv


Decomposed output written to: /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/decomposed_ssp_output_whirlpool.csv
Done: libya


## 3 · Cost-benefits

In [4]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
    output_path        = cfg.OUTPUT_CB_DATA,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

The TX TX:FRST:INCREASE_SEQUESTRATION is missing on AttTransformationCode
The TX TX:LNDU:BOUND_CLASSES is missing on AttTransformationCode
The TX TX:LNDU:DEC_CLASS_LOSS is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION is missing on AttTransformationCode
The TX TX:LNDU:BOUND_CLASSES is missing on AttTransformationCode
The TX TX:LNDU:DEC_CLASS_LOSS is missing on AttTransformationCode
The TX TX:ENFU:ADJ_EXPORTS is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT is missing on AttTransformationCode
The TX TX:ENFU:ADJ_EXPORTS is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT is missing on AttTransformationCode
The TX TX:ENFU:ADJ_EXPORTS is missing on AttTransformationCode
The TX TX:FRST:INCREASE_SEQUESTRATION is missing on AttTransformationCode
The TX TX:LNDU:BOUND_CLASSES is missing on AttTransformationCode
The TX TX:LNDU:DEC_CLASS_LOSS is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT is missing on AttTr

2026-04-10 15:31:05,435 - INFO - cb_data : (513016, 21)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/cost_benefits_data_whirlpool.csv


In [5]:
# ── Cargar cb_data desde CSV (si ya fue calculado) ───────────────────────────
# Ejecutar esta celda en lugar de la celda de arriba si cb_data ya fue guardado
import pandas as pd

cb_data = pd.read_csv(cfg.OUTPUT_CB_DATA)
logger.info(f"cb_data cargado : {cb_data.shape}  ← {cfg.OUTPUT_CB_DATA}")

2026-04-10 15:31:06,179 - INFO - cb_data cargado : (513016, 21)  ← /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/cost_benefits_data_whirlpool.csv


## 4 · Marginal Abatement Cost (MAC)

In [6]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    targets_path            = cfg.TARGETS_PATH,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_pflo_hble = cfg.STRATEGY_CODE_PFLO_HBLE,
    output_filename         = cfg.OUTPUT_MAC.name,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")

2026-04-10 15:31:06,441 - INFO - mac_df : (69, 9)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/marginal_abatement_costs_whirlpool.csv


## 5 · Tableau export — whirlpool

In [7]:
from mac_pipeline import build_attribute_map

att_map = build_attribute_map(
    att_primary          = att_primary,
    att_strategy         = att_strategy,
    run_output_dir       = cfg.RUN_ID_OUTPUT_DIR,
    tornado_att_map_path = cfg.RUN_OUTPUT_DIR / cfg.TORNADO_RUN_ID / "ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv",
)
logger.info(f"att_map : {att_map.shape}  → {cfg.RUN_ID_OUTPUT_DIR / 'ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv'}")

2026-04-10 15:31:06,451 - INFO - att_map : (69, 9)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-04-09T19;31;17.019336/ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv


In [8]:
from mac_pipeline import build_tableau_whirlpool

tableau_whirlpool = build_tableau_whirlpool(
    mac_df           = mac_df,
    run_output_dir   = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir      = cfg.TABLEAU_DIR,
    tornado_mac_path = cfg.TORNADO_MAC_PATH,
    output_path      = cfg.OUTPUT_TABLEAU_WHIRLPOOL,
)
logger.info(f"tableau_whirlpool : {tableau_whirlpool.shape}  → {cfg.OUTPUT_TABLEAU_WHIRLPOOL}")

2026-04-10 15:31:06,461 - INFO - tableau_whirlpool : (69, 19)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/Tableau/data/tableau_whirlpool.csv


## 6 · MAC tornado vs whirlpool

In [9]:
from mac_pipeline import build_mac_tornado_to_whirlpool

mac_comparison = build_mac_tornado_to_whirlpool(
    whirlpool_mac_df = mac_df,
    run_output_dir   = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir      = cfg.TABLEAU_DIR,
    tornado_mac_path = cfg.TORNADO_MAC_PATH,
    output_path      = cfg.OUTPUT_MAC_TORNADO_TO_WHIRLPOOL,
)
logger.info(f"mac_comparison : {mac_comparison.shape}  → {cfg.OUTPUT_MAC_TORNADO_TO_WHIRLPOOL}")

2026-04-10 15:31:06,468 - INFO - mac_comparison : (69, 11)  → /Users/fabianfuentes/git/ssp_libya/ssp_modeling/Tableau/data/mac_tornado_to_whirlpool.csv


## 7 · Summary

In [10]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",    "rows": len(df_decomposed), "cols": df_decomposed.shape[1], "file": cfg.OUTPUT_DECOMPOSED.name},
    {"output": "cost-benefits data","rows": len(cb_data),       "cols": cb_data.shape[1],       "file": cfg.OUTPUT_CB_DATA.name},
    {"output": "MAC curves",        "rows": len(mac_df),        "cols": mac_df.shape[1],        "file": cfg.OUTPUT_MAC.name},
])
print(summary.to_string(index=False))

            output   rows  cols                                   file
    decomposed SSP   1960  4107    decomposed_ssp_output_whirlpool.csv
cost-benefits data 513016    21       cost_benefits_data_whirlpool.csv
        MAC curves     69     9 marginal_abatement_costs_whirlpool.csv
